<a href="https://colab.research.google.com/github/Henibela/Neural-codes/blob/main/ae_light_model_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU name        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
import os

DRIVE_BASE    = '/content/drive/MyDrive/[04]Projects/Bsc-Thesis-Project/ae_lite'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Drive folder ready: {DRIVE_BASE}")
print(f"Checkpoint folder : {CHECKPOINT_DIR}")

In [ ]:
"""
ae_lite_model.py
================
AE-Lite: Lightweight Autoencoder for End-to-End Wireless Communication
Thesis: ML-Based Autoencoder for End-to-End Wireless Communication
Author: Henok Belayneh | Advisor: Dr. Tsegamlak Terefe
Addis Ababa University

Architecture (from design specification):
  Encoder: Linear(8,128) -> ReLU -> Linear(128,64) -> ReLU -> Linear(64,32) -> Power Norm
  Channel:  AWGN or Rayleigh flat fading
  Decoder: Linear(32,64) -> ReLU -> Linear(64,128) -> ReLU -> Linear(128,8) -> Sigmoid

Coding rate R = k/n = 8/16 = 0.5 bits per complex symbol.
The 32 real outputs of the encoder represent n=16 complex baseband symbols (real + imag pairs).
"""

import torch
import torch.nn as nn
import math


# =============================================================================
# ENCODER
# =============================================================================

class Encoder(nn.Module):
    """
    The transmitter side of the autoencoder.

    Takes a block of k=8 raw bits (as floats 0.0 / 1.0) and maps them
    to n=16 complex baseband symbols, represented as 2n=32 real numbers.

    After the final linear layer, a power normalisation step enforces
    the constraint that the average transmitted power per complex symbol
    equals 1.  Without this the network can "cheat" by making its signal
    arbitrarily large, which makes any BER-vs-SNR comparison meaningless.

    Layer sizes follow the design spec exactly:
        Input  :  8  (k bits)
        Hidden1: 128  neurons + ReLU
        Hidden2:  64  neurons + ReLU
        Output :  32  real values  (= 2n, n=16 complex symbols)
    """

    def __init__(self, k: int = 8, n: int = 16):
        """
        Args:
            k : number of input bits per message block  (default 8)
            n : number of complex symbols per block      (default 16)
                Output dimension will be 2*n real numbers.
        """
        super().__init__()

        self.k = k          # input bits
        self.n = n          # complex symbols out  (output = 2n reals)

        # --- Fully-connected layers ---
        # fc1: 8 inputs -> 128 neurons
        # Why 128?  Wide first layer gives the network enough "workspace"
        # to learn a rich feature mapping from 8 binary inputs.
        self.fc1 = nn.Linear(k, 128)

        # fc2: 128 -> 64
        # Compression step.  Forces the network to distil the most
        # important features — classic autoencoder bottleneck behaviour.
        self.fc2 = nn.Linear(128, 64)

        # fc3: 64 -> 32  (= 2n real values representing n complex symbols)
        # No activation after this layer; power normalisation comes next.
        self.fc3 = nn.Linear(64, 2 * n)

        # Shared ReLU activation.
        # ReLU(x) = max(0, x).  Applied after fc1 and fc2 to introduce
        # non-linearity — without it, stacking linear layers is still just
        # one linear transformation no matter how many layers you add.
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x : (batch_size, k) tensor of bits  — dtype float32, values 0. or 1.

        Returns:
            tx : (batch_size, 2n) tensor of power-normalised transmitted symbols.
        """
        # --- Layer 1 ---
        # x shape:  (batch, 8)
        x = self.relu(self.fc1(x))   # -> (batch, 128)

        # --- Layer 2 ---
        x = self.relu(self.fc2(x))   # -> (batch, 64)

        # --- Output layer (no activation yet) ---
        x = self.fc3(x)              # -> (batch, 32)

        # --- Power normalisation ---
        # Goal: enforce that the average power per complex symbol = 1.
        #
        # The 32 outputs are arranged as [Re(s0), Im(s0), Re(s1), Im(s1), ...]
        # The total signal energy in a block of n complex symbols is:
        #     E = sum_i ( Re(si)^2 + Im(si)^2 )  =  ||x||^2
        #
        # We want the *average* energy per symbol = 1, i.e. E/n = 1.
        # So we scale: x_norm = x  /  ( ||x|| / sqrt(n) )
        #
        # x.norm(dim=-1, keepdim=True) computes the L2 norm along the last
        # dimension for every sample in the batch, keeping shape (batch, 1)
        # so that division broadcasts correctly across the 32 columns.
        #
        # Adding 1e-8 prevents division by zero on pathological initialisation.
        tx = x / (x.norm(dim=-1, keepdim=True) / math.sqrt(self.n) + 1e-8)

        return tx   # -> (batch, 32),  avg symbol power = 1 guaranteed


# =============================================================================
# CHANNEL  (AWGN and Rayleigh)
# =============================================================================

class AWGNChannel(nn.Module):
    """
    Additive White Gaussian Noise channel.

    Adds complex Gaussian noise to the transmitted signal.  Because the
    signal is represented as 32 real numbers (not complex objects), noise
    is simply a real Gaussian tensor of the same shape.

    Noise standard deviation:
        sigma = sqrt( 1 / (2 * R * SNR_linear) )

    where R = k / n = 0.5 is the coding rate and SNR_linear = 10^(Eb/N0_dB / 10).

    This formula comes directly from the definition of Eb/N0:
        Eb/N0 = (signal energy per bit) / (noise spectral density)
    Rearranging for the noise variance gives the formula above.
    """

    def __init__(self, R: float = 0.5):
        """
        Args:
            R : coding rate = k/n  (default 0.5)
        """
        super().__init__()
        self.R = R

    def forward(self, tx: torch.Tensor, snr_db: float) -> torch.Tensor:
        """
        Args:
            tx     : (batch, 2n) power-normalised transmitted signal
            snr_db : Eb/N0 in dB

        Returns:
            rx : (batch, 2n) received signal = tx + noise
        """
        snr_linear = 10.0 ** (snr_db / 10.0)
        # Noise std dev from the Eb/N0 definition
        sigma = (1.0 / (2.0 * self.R * snr_linear)) ** 0.5
        # torch.randn_like creates a tensor of the same shape and device as tx
        # filled with samples from N(0,1), then scaled by sigma -> N(0, sigma^2)
        noise = torch.randn_like(tx) * sigma
        return tx + noise


class RayleighChannel(nn.Module):
    """
    Rayleigh flat-fading channel with coherent detection.

    Each transmitted block is multiplied by a complex fading coefficient
    h ~ CN(0, 1)  (circularly symmetric complex Gaussian).

    In the real-valued representation used here (32 real numbers for 16
    complex symbols), the fading is applied per-symbol pair:
        [Re(r_i), Im(r_i)] = h_real * [Re(s_i), Im(s_i)]
                            - h_imag * [Im(s_i), -Re(s_i)]
                            + noise

    For simplicity (and consistency with your Month 2-3 baseline) we
    draw one scalar fading coefficient per sample in the batch (i.e.
    flat fading — the entire block sees the same channel realisation).

    Coherent detection: the receiver is assumed to know |h|, and the
    received signal is equalised by dividing by |h| before decoding.
    This is the standard assumption in your baseline Rayleigh simulations.
    """

    def __init__(self, R: float = 0.5):
        super().__init__()
        self.R = R

    def forward(self, tx: torch.Tensor, snr_db: float) -> torch.Tensor:
        """
        Args:
            tx     : (batch, 2n) power-normalised transmitted signal
            snr_db : Eb/N0 in dB  (before fading)

        Returns:
            rx_eq : (batch, 2n) equalised received signal
        """
        batch = tx.size(0)
        device = tx.device

        # Fading coefficients: h = h_r + j*h_i,  h ~ CN(0,1)
        # Real and imaginary parts each ~ N(0, 1/2)
        h_r = torch.randn(batch, 1, device=device) / (2 ** 0.5)  # (batch, 1)
        h_i = torch.randn(batch, 1, device=device) / (2 ** 0.5)  # (batch, 1)
        # |h|^2  — used for equalisation
        h_mag2 = h_r ** 2 + h_i ** 2                              # (batch, 1)
        h_mag  = h_mag2 ** 0.5                                     # (batch, 1)

        # Apply fading to real and imaginary parts of each symbol.
        # tx is arranged as [Re0, Im0, Re1, Im1, ...]
        re = tx[:, 0::2]   # (batch, n) — real parts
        im = tx[:, 1::2]   # (batch, n) — imaginary parts

        # Complex multiplication: (h_r + j*h_i)(re + j*im)
        #   = h_r*re - h_i*im  +  j*(h_r*im + h_i*re)
        faded_re = h_r * re - h_i * im   # (batch, n)
        faded_im = h_r * im + h_i * re   # (batch, n)

        # Interleave back to (batch, 2n)
        faded = torch.zeros_like(tx)
        faded[:, 0::2] = faded_re
        faded[:, 1::2] = faded_im

        # Add AWGN noise
        snr_linear = 10.0 ** (snr_db / 10.0)
        sigma = (1.0 / (2.0 * self.R * snr_linear)) ** 0.5
        noise = torch.randn_like(faded) * sigma
        rx = faded + noise

        # Coherent equalisation: divide by |h| to undo fading amplitude
        # This gives the receiver a clean (but still noisy) signal.
        rx_eq = rx / (h_mag + 1e-8)

        return rx_eq


# =============================================================================
# DECODER
# =============================================================================

class Decoder(nn.Module):
    """
    The receiver side of the autoencoder.

    Takes the noisy (and possibly faded) received signal — 32 real numbers —
    and maps it back to estimated probabilities for the original k=8 bits.

    Architecture mirrors the encoder (expand back from narrow to wide):
        Input  :  32  (= 2n noisy received values)
        Hidden1:  64  neurons + ReLU
        Hidden2: 128  neurons + ReLU
        Output :   8  sigmoid units  (one probability per input bit)

    The Sigmoid output squashes each value to (0, 1), which is required
    by BCELoss (Binary Cross-Entropy Loss) used during training.
    Output[i] ≈ 1.0 means the decoder is confident bit i was 1.
    Output[i] ≈ 0.0 means the decoder is confident bit i was 0.
    """

    def __init__(self, k: int = 8, n: int = 16):
        """
        Args:
            k : number of output bit probabilities  (default 8)
            n : number of complex symbols received  (input = 2n reals)
        """
        super().__init__()

        self.k = k
        self.n = n

        # Decoder is a mirror of the encoder (narrow -> wide -> output)
        # fc1: 32 inputs -> 64 neurons
        self.fc1 = nn.Linear(2 * n, 64)

        # fc2: 64 -> 128
        self.fc2 = nn.Linear(64, 128)

        # fc3: 128 -> 8  (one output per input bit)
        self.fc3 = nn.Linear(128, k)

        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        # Sigmoid is ONLY applied at the final layer.
        # Why not ReLU?  Because ReLU output can exceed 1, but BCELoss
        # requires inputs strictly in (0,1).  Sigmoid guarantees this.

    def forward(self, y: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            y : (batch_size, 2n) noisy received signal

        Returns:
            probs : (batch_size, k) estimated bit probabilities in (0, 1)
        """
        # --- Layer 1 ---
        y = self.relu(self.fc1(y))      # -> (batch, 64)

        # --- Layer 2 ---
        y = self.relu(self.fc2(y))      # -> (batch, 128)

        # --- Output layer with Sigmoid ---
        probs = self.sigmoid(self.fc3(y))   # -> (batch, 8)

        return probs


# =============================================================================
# FULL AE-LITE MODEL  (Encoder + Channel + Decoder)
# =============================================================================

class AELite(nn.Module):
    """
    Complete end-to-end autoencoder transceiver.

    Wraps the Encoder, a chosen channel model, and the Decoder into one
    nn.Module that can be trained with a single optimizer call.

    The channel is included inside forward() so that PyTorch's autograd
    can differentiate through the entire pipeline.  Gradients flow back
    through the noise addition (which is just an addition — differentiable)
    into the decoder and then into the encoder.  Both are updated together
    in one backward() call — this is the "end-to-end" in end-to-end learning.

    Usage:
        model = AELite(channel='awgn')
        bits_hat = model(bits, snr_db=7.0)   # training
        bits_hat = model(bits, snr_db=12.0)  # evaluation at different SNR
    """

    def __init__(self, k: int = 8, n: int = 16, channel: str = 'awgn'):
        """
        Args:
            k       : bits per block   (default 8)
            n       : complex symbols  (default 16)  — coding rate R = k/n = 0.5
            channel : 'awgn' or 'rayleigh'
        """
        super().__init__()

        self.k = k
        self.n = n
        self.R = k / n   # coding rate = 0.5

        self.encoder = Encoder(k=k, n=n)
        self.decoder = Decoder(k=k, n=n)

        if channel == 'awgn':
            self.channel = AWGNChannel(R=self.R)
        elif channel == 'rayleigh':
            self.channel = RayleighChannel(R=self.R)
        else:
            raise ValueError(f"Unknown channel '{channel}'. Choose 'awgn' or 'rayleigh'.")

        self.channel_name = channel

    def forward(self, bits: torch.Tensor, snr_db: float) -> torch.Tensor:
        """
        Full forward pass: encode -> channel -> decode.

        Args:
            bits   : (batch_size, k) float tensor of input bits (0.0 or 1.0)
            snr_db : Eb/N0 in dB for this forward pass

        Returns:
            bits_hat : (batch_size, k) estimated bit probabilities in (0, 1)
        """
        # Step 1: Encode bits to transmitted symbols
        tx = self.encoder(bits)          # (batch, 2n),  avg power = 1

        # Step 2: Pass through channel (adds noise, or fading + noise)
        rx = self.channel(tx, snr_db)   # (batch, 2n),  noisy received signal

        # Step 3: Decode received signal to bit probability estimates
        bits_hat = self.decoder(rx)      # (batch, k),   values in (0, 1)

        return bits_hat

    def encode_only(self, bits: torch.Tensor) -> torch.Tensor:
        """
        Run only the encoder.  Useful for visualising the learned constellation.

        Args:
            bits : (batch_size, k) float tensor

        Returns:
            tx : (batch_size, 2n) transmitted symbols (power-normalised)
        """
        with torch.no_grad():
            return self.encoder(bits)

    def decode_only(self, rx: torch.Tensor) -> torch.Tensor:
        """
        Run only the decoder.  Useful for offline evaluation.

        Args:
            rx : (batch_size, 2n) received signal

        Returns:
            bits_hat : (batch_size, k) estimated bit probabilities
        """
        with torch.no_grad():
            return self.decoder(rx)

    def count_parameters(self) -> int:
        """Return total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def __repr__(self) -> str:
        return (
            f"AELite(k={self.k}, n={self.n}, R={self.R}, "
            f"channel='{self.channel_name}', "
            f"params={self.count_parameters():,})"
        )


In [ ]:
import random, time
import numpy as np
import matplotlib.pyplot as plt

# ═══════════════════════════════════════════════════════════════
# CONFIG — only edit things in this block between experiments
# ═══════════════════════════════════════════════════════════════
CHANNEL     = 'awgn'     # 'awgn' or 'rayleigh'
NUM_EPOCHS  = 10_000    # start here; increase later with resume
BATCH_SIZE  = 256
LR          = 1e-3
SNR_MIN, SNR_MAX = 0.0, 20.0
k, n        = 8, 16

# Paths — all go to Drive so they survive session resets
FINAL_MODEL = os.path.join(DRIVE_BASE, f'ae_lite_final_{CHANNEL}.pt')
LOSS_PLOT   = os.path.join(DRIVE_BASE, f'ae_lite_training_loss_{CHANNEL}.png')
# ═══════════════════════════════════════════════════════════════

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = AELite(k=k, n=n, channel=CHANNEL).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Training on : {device}")
print(f"Channel     : {CHANNEL}")
print(f"Epochs      : {NUM_EPOCHS}")
print(f"Params      : {model.count_parameters():,}")
print("-" * 40)

model.train()
losses = []
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    snr_db = random.uniform(SNR_MIN, SNR_MAX)
    bits   = torch.randint(0, 2, (BATCH_SIZE, k), device=device).float()

    optimizer.zero_grad()
    loss = criterion(model(bits, snr_db), bits)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 500 == 0:
        elapsed = time.time() - t0
        print(f"Epoch {epoch:6d}  loss: {loss.item():.5f}  time: {elapsed:.0f}s")

    if epoch % 2000 == 0:
        ckpt = os.path.join(CHECKPOINT_DIR, f'ae_lite_{CHANNEL}_epoch{epoch}.pt')
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                     'optimizer': optimizer.state_dict(), 'loss': loss.item()}, ckpt)
        print(f"  checkpoint → Drive: {ckpt}")

# Save final model to Drive
torch.save(model.state_dict(), FINAL_MODEL)
print(f"\nSaved: {FINAL_MODEL}")
print(f"Total time: {(time.time()-t0)/60:.1f} min")

# Loss curve — saved to Drive
w = max(1, NUM_EPOCHS // 100)
smooth = np.convolve(losses, np.ones(w)/w, mode='valid')
fig, ax = plt.subplots(figsize=(8,3.5))
ax.plot(losses, alpha=0.2, color='#1D9E75')
ax.plot(smooth, color='#1D9E75', lw=1.5, label='smoothed loss')
ax.axhline(np.log(2), color='gray', ls='--', lw=1, label='random baseline')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCELoss')
ax.set_title(f'Training loss — AE-Lite {CHANNEL.upper()}')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(LOSS_PLOT, dpi=150); plt.show()
print(f"Loss plot → Drive: {LOSS_PLOT}")

In [ ]:
from scipy.special import erfc

# Load the trained model from Drive
model.load_state_dict(torch.load(FINAL_MODEL, map_location=device))
model.eval()
print(f"Loaded: {FINAL_MODEL}")

SNR_RANGE  = np.arange(0, 21, 2)
N_BITS     = 1_000_000   # increase to 2_000_000 for thesis figures
EVAL_BATCH = 4096

def eval_ber(snr_db):
    errors = total = 0
    with torch.no_grad():
        for _ in range(N_BITS // (EVAL_BATCH * k)):
            b = torch.randint(0, 2, (EVAL_BATCH, k), device=device).float()
            errors += ((model(b, snr_db) > 0.5).float() != b).sum().item()
            total  += EVAL_BATCH * k
    return errors / total

ber_ae_awgn = []
print(f"{'SNR':>6}  {'BER':>12}"); print("-"*22)
for snr in SNR_RANGE:
    b = eval_ber(float(snr))
    ber_ae_awgn.append(b)
    print(f"{snr:>6.0f}  {b:>12.2e}")

ber_ae_awgn = np.array(ber_ae_awgn)
np.save(os.path.join(DRIVE_BASE, f'ber_ae_{CHANNEL}.npy'), ber_ae_awgn)
np.save(os.path.join(DRIVE_BASE, 'snr_range.npy'), SNR_RANGE)
print("BER arrays saved to Drive.")

In [ ]:
snr_fine = np.arange(0, 21, 0.25)
ber_theory_awgn = np.clip([0.5*erfc(np.sqrt(10**(s/10))) for s in snr_fine], 1e-7, 1)
ber_theory_ray  = np.clip([0.5*(1-np.sqrt((10**(s/10))/(1+10**(s/10)))) for s in snr_fine], 1e-7, 1)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.semilogy(snr_fine, ber_theory_awgn,  'k--', lw=1.3, label='BPSK/QPSK theory (AWGN)')
ax.semilogy(snr_fine, ber_theory_ray,   'b--', lw=1.3, label='BPSK theory (Rayleigh)')
valid = ber_ae_awgn > 1e-7
ax.semilogy(SNR_RANGE[valid], ber_ae_awgn[valid],
            's-', color='#1D9E75', lw=1.6, ms=6,
            label=f'AE-Lite ({CHANNEL.upper()}, simulated)')
ax.set_xlabel(r'$E_b/N_0$ (dB)', fontsize=13)
ax.set_ylabel('Bit Error Rate (BER)', fontsize=13)
ax.set_title(r'BER vs $E_b/N_0$ — AE-Lite vs BPSK/QPSK  ($R=0.5$)', fontsize=12)
ax.set_xlim(0, 20); ax.set_ylim(1e-5, 1)
ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.5); ax.legend(fontsize=10)
plt.tight_layout()
BER_PLOT = os.path.join(DRIVE_BASE, f'ae_lite_ber_comparison_{CHANNEL}.png')
plt.savefig(BER_PLOT, dpi=200, bbox_inches='tight'); plt.show()
print(f"BER plot → Drive: {BER_PLOT}")

# Constellation
all_msgs = torch.zeros(256, k)
for i in range(256):
    all_msgs[i] = torch.tensor([(i>>b)&1 for b in range(k)], dtype=torch.float32)
with torch.no_grad():
    tx_all = model.encoder(all_msgs.to(device)).cpu()
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(tx_all[:,0], tx_all[:,1], c=np.arange(256), cmap='tab20b', s=18, alpha=0.85)
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k--', lw=0.8, alpha=0.3)
ax.set_aspect('equal'); ax.set_title('Learned constellation'); ax.grid(alpha=0.25)
CONST_PLOT = os.path.join(DRIVE_BASE, f'ae_lite_constellation_{CHANNEL}.png')
plt.savefig(CONST_PLOT, dpi=150); plt.show()
print(f"Constellation → Drive: {CONST_PLOT}")

In [ ]:

CHANNEL       = 'awgn'
RESUME_EPOCH  = 2000     # the epoch number in your checkpoint filename
EXTRA_EPOCHS  = 8000     # how many MORE epochs to train
LR            = 1e-3     # keep same LR, or lower to 1e-4 if loss was plateauing

CKPT_PATH   = os.path.join(CHECKPOINT_DIR, f'ae_lite_{CHANNEL}_epoch{RESUME_EPOCH}.pt')
FINAL_MODEL = os.path.join(DRIVE_BASE, f'ae_lite_final_{CHANNEL}.pt')

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = AELite(channel=CHANNEL).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model'])
optimizer.load_state_dict(ckpt['optimizer'])
print(f"Resumed from epoch {ckpt['epoch']}, loss was {ckpt['loss']:.5f}")

model.train()
for epoch in range(1, EXTRA_EPOCHS + 1):
    snr_db = random.uniform(0, 20)
    bits   = torch.randint(0, 2, (256, 8), device=device).float()
    optimizer.zero_grad()
    loss = criterion(model(bits, snr_db), bits)
    loss.backward(); optimizer.step()
    if epoch % 500 == 0:
        abs_epoch = ckpt['epoch'] + epoch
        print(f"Epoch {abs_epoch}  loss: {loss.item():.5f}")
    if epoch % 2000 == 0:
        abs_epoch = ckpt['epoch'] + epoch
        p = os.path.join(CHECKPOINT_DIR, f'ae_lite_{CHANNEL}_epoch{abs_epoch}.pt')
        torch.save({'epoch':abs_epoch,'model':model.state_dict(),
                     'optimizer':optimizer.state_dict(),'loss':loss.item()}, p)

torch.save(model.state_dict(), FINAL_MODEL)
print(f"Saved: {FINAL_MODEL}")